# EDA — Predicting F1 Top-10 Finishes (2022–2024)

**Course:** IIT414W | **Group:** G19 | **Members:** Daniela Ávila, Klaus Krause

**Problem:** Predict whether a driver finishes in the Top 10 of a Formula 1 race.

**Data:** Race results 2022–2024 via the Jolpica API (`https://api.jolpi.ca/ergast/f1/`).

**Structure:** Every section follows `## Question → ### Data → ### Answer → ### Decision`.

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 414

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
from scipy import stats
from scipy.stats import spearmanr, pointbiserialr, chi2_contingency

np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 110

SEASONS = [2022, 2023, 2024]
BASE_URL = 'https://api.jolpi.ca/ergast/f1'

print(f'RANDOM_SEED = {RANDOM_SEED}')

In [ ]:
# ── Data loading ─────────────────────────────────────────────────────────────
def fetch_season_results(year):
    """Fetch all race results for one F1 season from the Jolpica API."""
    url = f'{BASE_URL}/{year}/results.json'
    r = requests.get(url, params={'limit': 1000}, timeout=30)
    r.raise_for_status()
    races = r.json()['MRData']['RaceTable']['Races']
    rows = []
    for race in races:
        for res in race['Results']:
            rows.append({
                # Identifiers
                'season':           int(race['season']),
                'round':            int(race['round']),
                'race_name':        race['raceName'],
                'circuit_id':       race['Circuit']['circuitId'],
                'circuit_name':     race['Circuit']['circuitName'],
                'country':          race['Circuit']['Location']['country'],
                # PRE-RACE features (available before the race — safe to use)
                'driver_id':        res['Driver']['driverId'],
                'driver_name':      res['Driver']['givenName'] + ' ' + res['Driver']['familyName'],
                'constructor_id':   res['Constructor']['constructorId'],
                'constructor_name': res['Constructor']['name'],
                'grid':             int(res['grid']),
                # POST-RACE outcomes (NOT features — leakage if used)
                'position_text':    res['positionText'],
                'status':           res['status'],
                'laps':             int(res['laps']),
                'points':           float(res['points']),
            })
    return rows

all_rows = []
for yr in SEASONS:
    print(f'Fetching {yr}...')
    all_rows.extend(fetch_season_results(yr))
    time.sleep(0.5)  # polite rate-limiting

df_raw = pd.DataFrame(all_rows)
print(f'\n{len(df_raw):,} driver-race entries | seasons: {sorted(df_raw["season"].unique())}')
print('Rounds per season:', df_raw.groupby('season')['round'].nunique().to_dict())

In [ ]:
# ── Target variable & derived columns ────────────────────────────────────────
# position_text: '1'–'20' for classified finishers; 'R'=retired, 'D'=DSQ, etc.
df_raw['position_num']   = pd.to_numeric(df_raw['position_text'], errors='coerce')
df_raw['top_10']         = (df_raw['position_num'] <= 10).astype(int)
df_raw['finished']       = df_raw['position_num'].notna().astype(int)
df_raw['pit_lane_start'] = (df_raw['grid'] == 0).astype(int)  # grid=0 means pit-lane start

df = df_raw.copy()
print('Dataset shape:', df.shape)
print('\nTarget distribution (%):')
print(df['top_10'].value_counts(normalize=True).rename({0: 'non-top-10', 1: 'top-10'}).mul(100).round(1))
df.head(3)

---

# Section 1 — Data Quality Audit

> Requirement 3.7: missing values (MCAR/MAR/MNAR classification for ≥3 columns), data types, outliers, temporal availability.

In [ ]:
# ── Missing values & data types ──────────────────────────────────────────────
print('=== DTYPES ===')
print(df.dtypes)

print('\n=== NULL COUNTS ===')
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print('No nulls in raw API columns.')
else:
    print(null_counts[null_counts > 0])

print('\n=== position_num NaNs (non-classified finishes) ===')
print(df['position_num'].isna().sum(), 'rows')
print('Status breakdown of DNF/DSQ entries:')
print(df[df['position_num'].isna()]['status'].value_counts().head(10))

print('\n=== grid distribution ===')
print(df['grid'].describe().round(2))
print(f'Pit-lane starters (grid=0): {df["pit_lane_start"].sum()}')

print('\n=== top_10 null check ===')
print('top_10 NaNs:', df['top_10'].isna().sum())

### Data Quality Findings

| Column | Issue | Classification | Decision |
|---|---|---|---|
| `position_num` | Non-numeric values (R, D, W, F, N) for retired/DSQ drivers | **MNAR** — retirement is correlated with race outcome (crash, mechanical failure, penalty); missing-ness is NOT random | Keep NaN; `top_10 = 0` for all non-classified rows (they did not finish in the top 10 by definition) |
| `grid` | Value = 0 for pit-lane starters (strategy/grid penalty) | **MAR** — grid=0 is a real event, not missing data; systematically different from grid=1–20 | Keep; flag via `pit_lane_start = 1`; exclude from grid-position analyses |
| `top_10` | Derived from `position_num`; no true NaNs after coercion | N/A | Target is complete and clean |
| `points` | Post-race outcome — **leakage risk** | N/A | Retained in DataFrame for reference only; **never used as feature** |
| `laps` | Post-race outcome — **leakage risk** | N/A | Same treatment as `points` |
| `status` | Post-race outcome (Finished / Retired / Collision / etc.) | N/A | Reference only; not a feature |

> **Temporal availability:** `grid`, `driver_id`, `constructor_id`, `circuit_id`, `season`, `round` are known **before the race starts** — safe as features. `position_num`, `points`, `laps`, `status` are known only **after** the race — using them as features causes leakage.

> Full log → `DATA_QUALITY_LOG.md`.

---

# Section 2 — Research Questions

> Requirement 3.1: ≥5 questions, each with Question → Plot → Interpretation → Decision.
> Structure: `## Question → ### Data → ### Answer → ### Decision`

## Q1 — Is the dataset balanced? (Class Balance Analysis)

**Question:** What is the distribution of Top-10 vs. non-Top-10 finishes? Is accuracy a safe evaluation metric? What would a naive always-predict-one-class baseline score?

In [ ]:
### Data
counts = df['top_10'].map({1: 'Top-10', 0: 'Non-Top-10'}).value_counts()
percentages = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(counts.index, counts.values,
            color=['#2196F3', '#FF7043'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Top-10 vs Non-Top-10 Finish Counts (2022–2024)', fontsize=12)
axes[0].set_ylabel('Number of driver-race entries')
axes[0].set_xlabel('Finish category')
for i, (lbl, val) in enumerate(counts.items()):
    axes[0].text(i, val + 8, f'{val}\n({percentages[lbl]:.1f}%)',
                 ha='center', va='bottom', fontsize=11)

axes[1].pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=['#2196F3', '#FF7043'], startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Balance (2022–2024)', fontsize=12)

plt.suptitle('Class Balance Analysis — Target Variable', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Always-predict-Top-10 accuracy:     {percentages["Top-10"]:.1f}%')
print(f'Always-predict-Non-Top-10 accuracy: {percentages["Non-Top-10"]:.1f}%')

### Answer

The dataset is **nearly perfectly balanced** (~50% Top-10, ~50% non-Top-10). This is structurally expected: exactly 10 of the ~20 drivers per race finish in the top 10. The small deviation from 50/50 comes from occasional disqualifications or fewer-than-20-starter races.

A naive baseline that **always predicts Top-10** achieves ~50% accuracy — the theoretical floor.

### Decision

Accuracy is a **reasonable metric** for this balanced dataset. Any model or heuristic scoring ≤ 50% adds no value over random guessing. We establish 50% as the minimum bar. This is evaluated formally in `baseline.ipynb`.

## Q2 — Is the Top-10 rate stable across the 2022, 2023, and 2024 seasons? (Temporal Pattern Analysis)

**Question:** Does the class distribution or feature distribution shift between seasons? This determines whether a temporal split produces a representative validation set.

In [ ]:
### Data
season_stats = df.groupby('season').agg(
    total=('top_10', 'count'),
    top10=('top_10', 'sum'),
    top10_rate=('top_10', 'mean'),
    races=('round', 'nunique'),
    drivers=('driver_id', 'nunique'),
    constructors=('constructor_id', 'nunique')
).reset_index()
season_stats['top10_pct'] = season_stats['top10_rate'] * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_s = ['#4CAF50', '#2196F3', '#FF9800']

axes[0].bar(season_stats['season'].astype(str), season_stats['top10_pct'],
            color=colors_s, edgecolor='white', linewidth=1.5)
axes[0].axhline(50, color='red', linestyle='--', alpha=0.6, label='50% floor')
axes[0].set_title('Top-10 Rate per Season', fontsize=12)
axes[0].set_ylabel('Top-10 finish rate (%)')
axes[0].set_xlabel('Season')
axes[0].set_ylim(40, 60)
axes[0].legend()
for i, row in season_stats.iterrows():
    axes[0].text(i, row['top10_pct'] + 0.3, f"{row['top10_pct']:.1f}%",
                 ha='center', va='bottom', fontsize=11)

grid_by_season = df[df['pit_lane_start'] == 0].groupby('season')['grid'].mean()
axes[1].bar(grid_by_season.index.astype(str), grid_by_season.values,
            color=colors_s, edgecolor='white', linewidth=1.5)
axes[1].set_title('Mean Grid Position per Season', fontsize=12)
axes[1].set_ylabel('Mean starting grid position')
axes[1].set_xlabel('Season')
for i, (s, v) in enumerate(grid_by_season.items()):
    axes[1].text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom', fontsize=11)

axes[2].bar(season_stats['season'].astype(str), season_stats['total'],
            color=colors_s, edgecolor='white', linewidth=1.5)
axes[2].set_title('Total Driver-Race Entries per Season', fontsize=12)
axes[2].set_ylabel('Driver-race entries')
axes[2].set_xlabel('Season')
for i, row in season_stats.iterrows():
    axes[2].text(i, row['total'] + 5, f"{row['total']}\n({row['races']} races)",
                 ha='center', va='bottom', fontsize=9)

plt.suptitle('Temporal Pattern Analysis — 2022 vs 2023 vs 2024', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(season_stats[['season','races','total','top10_pct','drivers','constructors']].to_string(index=False))

### Answer

The Top-10 rate is **stable across all three seasons** (~50% each year). Mean grid position is consistent (~10.5), indicating no systematic shift in grid distributions. 2024 has slightly more races due to calendar expansion.

### Decision

A **temporal split is appropriate and safe** — no distribution shift that would make a 2022–2023-trained heuristic invalid for 2024. Split defined in Section 5.

## Q3 — Does grid position predict a Top-10 finish?

**Question:** Is there a monotonic relationship between starting grid position and probability of finishing in the Top 10? This is the foundation of our baseline heuristic.

In [ ]:
### Data
df_grid = df[df['pit_lane_start'] == 0].copy()  # exclude pit-lane starters for this analysis

grid_top10 = df_grid.groupby('grid').agg(
    top10_rate=('top_10', 'mean'),
    count=('top_10', 'count')
).reset_index()
grid_top10['top10_pct'] = grid_top10['top10_rate'] * 100

rho, p_val = spearmanr(df_grid['grid'], df_grid['top_10'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = ['#2196F3' if g <= 10 else '#FF7043' for g in grid_top10['grid']]
axes[0].bar(grid_top10['grid'], grid_top10['top10_pct'],
            color=bar_colors, edgecolor='white', linewidth=0.8)
axes[0].axvline(10.5, color='black', linestyle='--', alpha=0.5)
axes[0].axhline(50, color='red', linestyle=':', alpha=0.5)
axes[0].set_title(f'Top-10 Finish Rate by Grid Position\nSpearman ρ = {rho:.3f}  (p < 0.001)', fontsize=12)
axes[0].set_xlabel('Grid position (1 = pole position)')
axes[0].set_ylabel('Top-10 finish rate (%)')
axes[0].set_xticks(range(1, 21))
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(fc='#2196F3', label='Grid 1–10'),
    Patch(fc='#FF7043', label='Grid 11–20'),
])

rolling = grid_top10.set_index('grid')['top10_pct'].rolling(window=3, center=True, min_periods=1).mean()
axes[1].plot(rolling.index, rolling.values, 'o-', color='#1565C0', linewidth=2, markersize=6)
axes[1].fill_between(grid_top10['grid'],
                      grid_top10['top10_pct'] - 5,
                      grid_top10['top10_pct'] + 5,
                      alpha=0.15, color='#1565C0', label='±5 pp band')
axes[1].axvline(10.5, color='black', linestyle='--', alpha=0.5, label='Grid 10/11 boundary')
axes[1].axhline(50, color='red', linestyle=':', alpha=0.5, label='50% floor')
axes[1].set_title('Smoothed Trend: Top-10 Rate vs Grid', fontsize=12)
axes[1].set_xlabel('Grid position')
axes[1].set_ylabel('Top-10 finish rate (%)')
axes[1].set_xticks(range(1, 21))
axes[1].legend()

plt.suptitle('Grid Position as Predictor of Top-10 Finish', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Spearman ρ = {rho:.4f}   p = {p_val:.2e}')
print(f'Top-10 rate, grid 1–10:  {df_grid[df_grid["grid"]<=10]["top_10"].mean()*100:.1f}%')
print(f'Top-10 rate, grid 11–20: {df_grid[df_grid["grid"]>10]["top_10"].mean()*100:.1f}%')

### Answer

Grid position has a **strong negative Spearman correlation** with Top-10 finish probability (ρ ≈ −0.50, p < 0.001). Drivers starting 1–10 finish in the top 10 ~70–80% of the time; those starting 11–20 do so only ~20–30% of the time. The relationship is monotonic but not deterministic — positions 9–12 show high variance, so upsets are common at the boundary.

### Decision

**Grid position is the single strongest pre-race predictor.** The rule "if grid ≤ 10 → predict top-10" is the natural domain heuristic. We will evaluate this threshold in `baseline.ipynb`.

## Q4 — Which constructors are most likely to place drivers in the Top 10?

**Question:** Do certain teams systematically outperform their grid position? Does constructor identity add independent predictive signal beyond grid?

In [ ]:
### Data
constructor_stats = df.groupby('constructor_name').agg(
    top10_rate=('top_10', 'mean'),
    entries=('top_10', 'count'),
    mean_grid=('grid', 'mean')
).reset_index().sort_values('top10_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

cmap_c = plt.cm.RdYlGn
norm_c = plt.Normalize(0, 1)
bar_c  = [cmap_c(norm_c(v)) for v in constructor_stats['top10_rate']]

bars = ax.barh(constructor_stats['constructor_name'],
               constructor_stats['top10_rate'] * 100,
               color=bar_c, edgecolor='white', linewidth=1)
ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='50% reference')
ax.set_title('Top-10 Finish Rate by Constructor (2022–2024)', fontsize=13)
ax.set_xlabel('Top-10 finish rate (%)')
ax.set_ylabel('Constructor')
ax.legend()
ax.set_xlim(0, 120)

for bar, (_, row) in zip(bars, constructor_stats.iterrows()):
    ax.text(bar.get_width() + 0.8,
            bar.get_y() + bar.get_height() / 2,
            f"{row['top10_rate']*100:.0f}%  ({row['entries']} starts, mean grid {row['mean_grid']:.1f})",
            va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

### Answer

Constructor performance is **highly stratified**: top teams (Red Bull, Mercedes, Ferrari) achieve top-10 rates well above 50%; backmarkers (Haas, Williams) stay consistently below. The gap spans ~40–60 percentage points across constructors. Comparing teams with similar mean grid positions confirms that constructor-specific factors (car performance, reliability, strategy) add **independent signal** beyond grid alone.

### Decision

**Constructor identity is the most valuable additional feature** for Lab 2. Our current grid-based heuristic implicitly captures some constructor effect (good teams qualify front), but explicitly encoding constructor improves prediction especially for mid-field uncertainty.

## Q5 — Are individual drivers consistently Top-10 performers across seasons?

**Question:** Do some drivers systematically over- or under-perform their grid position year-over-year? Would driver history add predictive signal beyond constructor?

In [ ]:
### Data
driver_stats = df.groupby('driver_name').agg(
    top10_rate=('top_10', 'mean'),
    entries=('top_10', 'count'),
    mean_grid=('grid', 'mean')
).reset_index()
driver_stats = driver_stats[driver_stats['entries'] >= 20].sort_values('top10_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
cmap_d = plt.cm.RdYlGn
bar_d  = [cmap_d(norm_c(v)) for v in driver_stats['top10_rate']]

bars_d = ax.barh(driver_stats['driver_name'],
                 driver_stats['top10_rate'] * 100,
                 color=bar_d, edgecolor='white', linewidth=1)
ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='50% reference')
ax.set_title('Top-10 Finish Rate by Driver\n(drivers with ≥20 starts, 2022–2024)', fontsize=12)
ax.set_xlabel('Top-10 finish rate (%)')
ax.set_ylabel('Driver')
ax.legend()
ax.set_xlim(0, 110)

for bar, (_, row) in zip(bars_d, driver_stats.iterrows()):
    ax.text(bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{row['top10_rate']*100:.0f}%",
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Year-on-year stability
driver_season_rate = df.groupby(['driver_name', 'season'])['top_10'].mean().unstack()
consistency_std = driver_season_rate.std(axis=1).dropna().sort_values()
print('Most consistent drivers (low std across seasons):')
print(consistency_std.head(5).round(3))
print('\nLeast consistent:')
print(consistency_std.tail(5).round(3))

### Answer

Driver top-10 rates range from ~10% to ~90% and are **stable across seasons** — elite drivers maintain high rates year-over-year. Low within-driver season-to-season standard deviations confirm this stability. However, driver performance is **highly correlated with constructor**: the best drivers drive for the best teams, making it difficult to isolate individual skill from car quality.

### Decision

**Driver identity adds signal but is partly redundant with constructor.** In Lab 2 we will test the marginal gain of adding driver history on top of constructor. We also document the survivorship bias in driver rates in Section 4 (Trap Awareness).

## Q6 — Has constructor dominance shifted across the 2022–2024 seasons?

**Question:** Did the competitive ordering between teams change between seasons? Temporal shifts in constructor performance matter for whether training on 2022–2023 generalizes to 2024.

In [ ]:
### Data
top_constructors = (
    df.groupby('constructor_name')['top_10'].count()
    .nlargest(8).index.tolist()
)

const_season = (
    df[df['constructor_name'].isin(top_constructors)]
    .groupby(['constructor_name', 'season'])['top_10'].mean()
    .reset_index()
)
const_pivot = (
    const_season
    .pivot(index='constructor_name', columns='season', values='top_10')
    .fillna(0)
    .sort_values(by=2024, ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    const_pivot * 100, annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=0, vmax=100, ax=axes[0],
    cbar_kws={'label': 'Top-10 rate (%)'}
)
axes[0].set_title('Constructor Top-10 Rate by Season (%)', fontsize=12)
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Constructor')

for constructor in const_pivot.index:
    axes[1].plot(
        const_pivot.columns, const_pivot.loc[constructor] * 100,
        marker='o', linewidth=2, label=constructor
    )
axes[1].axhline(50, color='red', linestyle=':', alpha=0.5, label='50% ref')
axes[1].set_title('Constructor Top-10 Rate Trend (2022–2024)', fontsize=12)
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Top-10 finish rate (%)')
axes[1].set_xticks([2022, 2023, 2024])
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.suptitle('Constructor Dominance Shift Across Seasons', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Answer

Constructor dominance is **partially stable** but shows meaningful shifts: Red Bull peaked in 2023, Mercedes declined from 2022 levels, McLaren rose toward 2024. This confirms a **moderate distribution shift** between training (2022–2023) and test (2024) data.

### Decision

This reinforces the need for a **temporal split** (not random). For Lab 2, constructor features should use recent-season rates rather than all-time averages to account for performance evolution.

---

# Section 3 — Correlation Analysis

> Requirement 3.4: Correlation for ≥5 candidate features. Using appropriate measures (Spearman for ordinal, Cramér's V for categorical, point-biserial for binary). Interpreting magnitude AND direction.

In [ ]:
### Data
def cramers_v(x, y):
    """Cramér's V: association between two categorical variables."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.to_numpy().sum()
    r, k = ct.shape
    return float(np.sqrt(chi2 / n / (min(r, k) - 1))) if min(r, k) > 1 else 0.0

df_corr = df[df['pit_lane_start'] == 0].copy()

rho_grid,   p_grid   = spearmanr(df_corr['grid'],   df_corr['top_10'])
rho_season, p_season = spearmanr(df_corr['season'], df_corr['top_10'])
cv_constructor = cramers_v(df_corr['constructor_id'], df_corr['top_10'])
cv_driver      = cramers_v(df_corr['driver_id'],      df_corr['top_10'])
cv_circuit     = cramers_v(df_corr['circuit_id'],     df_corr['top_10'])
rpb_pit, p_pit = pointbiserialr(df['pit_lane_start'], df['top_10'])

corr_table = pd.DataFrame([
    {'Feature': 'grid',           'Type': 'Numeric',     'Measure': 'Spearman ρ',    'Value': round(rho_grid, 4),       'Notes': 'Negative: lower grid → more top-10s'},
    {'Feature': 'season',         'Type': 'Numeric',     'Measure': 'Spearman ρ',    'Value': round(rho_season, 4),     'Notes': '~0: no year-over-year trend'},
    {'Feature': 'constructor_id', 'Type': 'Categorical', 'Measure': "Cramér's V",   'Value': round(cv_constructor, 4), 'Notes': 'Moderate-strong: team matters'},
    {'Feature': 'driver_id',      'Type': 'Categorical', 'Measure': "Cramér's V",   'Value': round(cv_driver, 4),      'Notes': 'Strong; correlated with constructor'},
    {'Feature': 'circuit_id',     'Type': 'Categorical', 'Measure': "Cramér's V",   'Value': round(cv_circuit, 4),     'Notes': 'Weak: circuit adds limited signal over grid'},
    {'Feature': 'pit_lane_start', 'Type': 'Binary',      'Measure': 'Point-biserial', 'Value': round(rpb_pit, 4),       'Notes': 'Negative: pit-lane starters rarely top-10'},
])
print(corr_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
vals_abs = corr_table['Value'].abs()
cmap_b   = plt.cm.Blues
colors_b = [cmap_b(0.35 + 0.55 * v / vals_abs.max()) for v in vals_abs]
ax.bar(corr_table['Feature'], vals_abs, color=colors_b, edgecolor='white', linewidth=1.5)
ax.set_title("Feature–Target Association Strength\n(|Spearman ρ| or Cramér's V)", fontsize=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Association magnitude')
ax.set_ylim(0, 1)
for i, (feat, val, orig) in enumerate(zip(corr_table['Feature'], vals_abs, corr_table['Value'])):
    sign = '' if orig >= 0 else '−'
    ax.text(i, val + 0.01, f'{sign}{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

### Correlation Analysis Summary

| Feature | Measure | Value | Direction | Interpretation |
|---|---|---|---|---|
| `grid` | Spearman ρ | ~−0.50 | **Negative** | Strong: lower grid → more top-10 finishes. Primary predictor. |
| `season` | Spearman ρ | ~0.00 | Negligible | No year-over-year trend in outcome. |
| `constructor_id` | Cramér's V | ~0.40 | N/A | Moderate–strong: team performance matters. |
| `driver_id` | Cramér's V | ~0.50 | N/A | Strong but partly redundant with constructor. |
| `circuit_id` | Cramér's V | ~0.10–0.15 | N/A | Weak: circuits add little over grid. |
| `pit_lane_start` | Point-biserial | ~−0.10 | **Negative** | Pit-lane starters rarely finish top-10. |

**Key takeaway:** `grid` dominates as the single strongest numerical predictor. `driver_id` and `constructor_id` add categorical signal but are correlated. `season` and `circuit_id` are weak features. This ranking informs Lab 2 feature selection.

---

# Section 4 — Trap Awareness: Survivorship Bias in Driver Historical Rates

> Requirement 3.5: Explicitly check for one of the three traps (spurious correlation, survivorship bias, anchoring bias).

**Trap being checked:** *Survivorship bias* when using driver historical top-10 rates as a feature.

**Hypothesis:** Drivers who consistently underperform tend to lose their F1 seats faster. Our 2022–2024 dataset therefore over-represents drivers who were *good enough to keep their seats*, biasing upward their observed top-10 rates. Drivers present across all 3 seasons are a self-selected high-performance group.

In [ ]:
### Data — survivorship bias check
driver_n_seasons = df.groupby('driver_id')['season'].nunique().reset_index()
driver_n_seasons.columns = ['driver_id', 'seasons_present']

driver_rates = df.groupby('driver_id')['top_10'].mean().reset_index()
driver_rates.columns = ['driver_id', 'top10_rate']

driver_analysis = driver_n_seasons.merge(driver_rates, on='driver_id')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

season_counts = driver_analysis['seasons_present'].value_counts().sort_index()
axes[0].bar(
    season_counts.index.astype(str), season_counts.values,
    color=['#FF7043', '#FFA726', '#66BB6A'], edgecolor='white', linewidth=1.5
)
axes[0].set_title('Seasons Per Driver in Dataset (2022–2024)', fontsize=12)
axes[0].set_xlabel('Number of seasons present (out of 3)')
axes[0].set_ylabel('Number of drivers')
for i, (s, v) in enumerate(season_counts.items()):
    axes[0].text(i, v + 0.2, str(v), ha='center', va='bottom', fontsize=12)

for n_s, group in driver_analysis.groupby('seasons_present'):
    axes[1].scatter(
        [n_s] * len(group), group['top10_rate'] * 100,
        alpha=0.6, s=60, label=f'{n_s} season(s)'
    )
means_surv = driver_analysis.groupby('seasons_present')['top10_rate'].mean()
axes[1].plot(means_surv.index, means_surv.values * 100, 'k--o',
             linewidth=2, markersize=8, label='Group mean')
axes[1].set_title('Top-10 Rate vs. Seasons Present\n(survivorship bias check)', fontsize=12)
axes[1].set_xlabel('Seasons driver appears in dataset')
axes[1].set_ylabel('Top-10 finish rate (%)')
axes[1].set_xticks([1, 2, 3])
axes[1].legend()

plt.suptitle('Survivorship Bias — Driver Selection Effect', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Mean top-10 rate by seasons present:')
print((driver_analysis.groupby('seasons_present')['top10_rate'].mean() * 100).round(1))

### What We Found

Drivers present in **all 3 seasons** have systematically higher top-10 rates than drivers present in only 1–2 seasons. This is consistent with survivorship bias: low performers lose their seats and are replaced by rookies who (on average) perform below the level of the multi-season veterans.

**Implication:** Using raw driver historical top-10 rates as features produces an upward-biased estimate for long-tenured drivers, and no estimate at all for first-season drivers (cold-start problem).

**Trap confirmed.** Using driver rates without survivorship correction would inflate apparent driver-level feature importance. For Lab 2: use recency-weighted rates, add a rookie flag, or rely on constructor rather than driver to avoid the cold-start problem.

---

# Section 5 — Temporal Train / Validation / Test Split

> Requirement 3.6: Explicit temporal split with written rationale. No random splits.

## Split Design

| Set | Data | Purpose |
|---|---|---|
| **Train** | 2022 + 2023 full seasons | Learn historical patterns — two complete regulatory years |
| **Validation** | 2024 rounds 1 – (N/2) | Evaluate and tune baseline; recent but not \"future\" data |
| **Test** | 2024 rounds (N/2+1) – end | **Held out entirely** — untouched until Lab 2 final evaluation |

**Why temporal and not random?** F1 data is time-ordered. A random split would leak future information: a training row might come from 2024 while a validation row comes from 2022, meaning the model has seen 2024 performance during training. This produces optimistically biased accuracy that does not reflect real prediction performance. The temporal split simulates the actual deployment scenario: train on the past, predict the future.

In [ ]:
### Define split
total_rounds_2024 = int(df[df['season'] == 2024]['round'].max())
val_end   = total_rounds_2024 // 2
test_start = val_end + 1

print(f'Total 2024 rounds: {total_rounds_2024}')
print(f'Validation: 2024 rounds 1–{val_end}')
print(f'Test:       2024 rounds {test_start}–{total_rounds_2024}')

train_mask = df['season'].isin([2022, 2023])
val_mask   = (df['season'] == 2024) & (df['round'] <= val_end)
test_mask  = (df['season'] == 2024) & (df['round'] >= test_start)

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print('\n=== SPLIT SUMMARY ===')
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{name:6s}: {len(split):5,} rows | {split["round"].nunique():2d} races '
          f'| top-10 rate: {split["top_10"].mean()*100:.1f}%')

# ── Leakage checks ───────────────────────────────────────────────────────────
assert df_train['season'].max() <= 2023,    'LEAKAGE: 2024 data in train!'
assert df_val['season'].eq(2024).all(),     'Val should only contain 2024'
assert df_val['round'].max() <= val_end,    'LEAKAGE: test rounds in val!'
assert df_test['round'].min() >= test_start,'LEAKAGE: val rounds in test!'
assert len(set(df_val.index) & set(df_test.index)) == 0, 'Overlapping val/test!'
print('\n✓ All leakage checks passed.')

In [ ]:
# Visualize the split
df['_split'] = 'Unknown'
df.loc[train_mask, '_split'] = 'Train (2022–2023)'
df.loc[val_mask,   '_split'] = 'Validation (2024 H1)'
df.loc[test_mask,  '_split'] = 'Test (2024 H2) — HELD OUT'

df_plot = df[['season', 'round', '_split']].copy()
df_plot['x'] = df_plot['season'] + (df_plot['round'] - 1) / 26

fig, ax = plt.subplots(figsize=(12, 3))
colors_split = {
    'Train (2022–2023)':        '#4CAF50',
    'Validation (2024 H1)':     '#2196F3',
    'Test (2024 H2) — HELD OUT':'#FF7043',
}
for label, color in colors_split.items():
    subset = df_plot[df_plot['_split'] == label]
    ax.scatter(subset['x'], [label] * len(subset), color=color, s=25, alpha=0.5)

ax.set_title('Temporal Train / Validation / Test Split', fontsize=13)
ax.set_xlabel('Season (approximate position along timeline)')
ax.set_yticks(list(colors_split.keys()))
ax.set_xlim(2021.8, 2025.0)
ax.axvline(2024, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Clean up helper column
df.drop(columns=['_split'], inplace=True)

### Split Rationale Summary

- **Train (2022–2023):** Two full seasons — sufficient history, covers two regulatory eras.
- **Validation (2024 H1):** Used exclusively to evaluate baseline accuracy in `baseline.ipynb`. Reflects current-season car performance.
- **Test (2024 H2):** Completely held out. **Never used in any analysis, feature selection, or threshold tuning.** Reserved for Lab 2 final evaluation.

All three splits maintain a ~50% top-10 rate, confirming no label leakage.

---

# Section 6 — 1-3-1 Summary

> Requirement 3.8: Final Markdown cell with 1-3-1 summary. This is the same text submitted on Canvas.

---

## Grid position is the dominant pre-race signal — build the baseline around it, and plan to add constructor in Lab 2.

**Three evidence points:**

1. **Grid position has the strongest pre-race correlation with Top-10 finish** (Spearman ρ ≈ −0.50, p < 0.001): drivers starting 1–10 finish in the top 10 ~70–80% of the time, while those starting 11–20 do so only ~20–30% of the time. This single feature makes a rule-based heuristic viable.

2. **Constructor identity is the strongest additional signal** (Cramér's V ≈ 0.40): top teams achieve top-10 rates ~70–100% regardless of grid, while backmarkers stay below 30% even from mid-field starts. The gap spans ~50 percentage points. Constructor dominance is stable across 2022–2024 (with some mid-field shifts), making it a reliable feature for Lab 2 while confirming the need for a temporal — not random — split.

3. **Survivorship bias inflates driver historical top-10 rates**: drivers present across all 3 seasons show ~10–15 percentage points higher top-10 rates than one-season drivers because low performers lose their seats. Using raw driver rates as features without bias correction would overstate driver-level predictive power and fail entirely on rookies (cold-start problem).

**Action:** Implement "grid ≤ 10 → predict Top-10" as the domain heuristic baseline in `baseline.ipynb`, evaluated on the 2024 H1 validation set. Keep 2024 H2 as an untouched test set. In Lab 2, add constructor-level encoding as the primary feature enrichment and apply survivorship-bias correction before including driver historical rates.